<a href="https://colab.research.google.com/github/sangjkim930/AI-Driven-Research-Methodology/blob/main/02_In_Memory_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Simple In-Memory RAG System

**PDFs → Text → Chunks → Embeddings → Relevant Evidence → Grounded Answer**

Run the cells in order from top to bottom.

### Step 1. Set Up the Environment

Install the required libraries, connect to the OpenAI API, and define the research question.

In [10]:
!pip install -q openai pypdf numpy

In [11]:
import glob
import os
import re
import numpy as np

from google.colab import userdata
from openai import OpenAI
from pypdf import PdfReader

client = OpenAI(
    api_key=userdata.get("OPENAI_API_KEY")
)

EMBED_MODEL = "text-embedding-3-small"
ANSWER_MODEL = "gpt-4o-mini"

CHUNK_SIZE = 1800
CHUNK_OVERLAP = 250
TOP_K = 10

QUESTION = (
    "Based on the uploaded papers' own empirical results, "
    "which papers report a nonlinear relationship between "
    "environmental performance and financial performance?"
)

### Step 2. Load and Prepare the Papers

Upload the PDF papers using the **Files** panel on the left side of Google Colab. Then run the code below.

In [ ]:
pdf_files = sorted(glob.glob("/content/*.pdf"))

print("Papers found:", len(pdf_files))
for file_name in pdf_files:
    print("-", os.path.basename(file_name))

In [ ]:
# Extract text page by page

pages = []

for pdf_path in pdf_files:
    reader = PdfReader(pdf_path)
    in_references = False

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""

        # Skip the reference section once it begins
        if re.search(r"(?im)^\s*(references|bibliography)\s*$", text):
            in_references = True

        if in_references:
            continue

        text = re.sub(r"\s+", " ", text).strip()

        if text:
            pages.append({
                "paper": pdf_path,
                "page": page_number,
                "text": text
            })

print("Pages extracted:", len(pages))

### Step 3. Create Searchable Text

Divide the papers into smaller chunks and convert the chunks into embeddings.

In [ ]:
# Divide pages into overlapping chunks

chunks = []

for page in pages:
    text = page["text"]
    start = 0

    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))
        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append({
                "paper": page["paper"],
                "page": page["page"],
                "text": chunk_text
            })

        if end == len(text):
            break

        start = end - CHUNK_OVERLAP

print("Chunks created:", len(chunks))

In [ ]:
# Create embeddings for all chunks

chunk_texts = [chunk["text"] for chunk in chunks]

response = client.embeddings.create(
    model=EMBED_MODEL,
    input=chunk_texts
)

embedding_matrix = np.array(
    [item.embedding for item in response.data],
    dtype=np.float32
)

# Normalize vectors so dot product equals cosine similarity
embedding_matrix /= np.linalg.norm(
    embedding_matrix,
    axis=1,
    keepdims=True
)

print("Embedding matrix:", embedding_matrix.shape)

### Step 4. Retrieve Relevant Evidence

Use the research question itself to find the most relevant passages from the uploaded papers.

In [7]:
# Retrieve the most relevant evidence

def looks_like_reference_chunk(text):
    """
    Identify chunks that appear to consist mainly
    of bibliography or reference-list entries.

    This is a heuristic rather than a perfect classifier.
    """

    lower_text = text.lower().strip()

    # References or Bibliography near the beginning
    has_reference_heading = bool(
        re.search(
            r"\b(references|bibliography)\b",
            lower_text[:200]
        )
    )

    # Many publication years in one chunk
    years = re.findall(
        r"\b(?:19|20)\d{2}[a-z]?\b",
        text
    )

    # Repeated DOI strings
    doi_count = lower_text.count(
        "doi"
    )

    return (
        has_reference_heading
        or len(years) >= 12
        or doi_count >= 4
    )

def retrieve_chunks(
    query,
    k,
    max_per_paper,
    filter_reference_material
):
    """
    Convert the query into an embedding,
    calculate cosine similarity, and retrieve
    the highest-ranking chunks.
    """

    query_response = client.embeddings.create(
        model=EMBED_MODEL,
        input=query
    )

    query_vector = np.asarray(
        query_response.data[0].embedding,
        dtype=np.float32
    )

    query_vector = query_vector / max(
        np.linalg.norm(query_vector),
        1e-12
    )

    # Because all vectors are normalized,
    # dot product equals cosine similarity.
    scores = (
        embedding_matrix
        @ query_vector
    )

    ranked_indices = np.argsort(
        scores
    )[::-1]

    selected = []
    paper_counts = defaultdict(int)

    for index in ranked_indices:
        index = int(index)

        chunk = chunks[index]

        if filter_reference_material:
            if chunk.get(
                "is_reference_page",
                False
            ):
                continue

            if looks_like_reference_chunk(
                chunk["text"]
            ):
                continue

        paper = chunk["paper"]

        if (
            paper_counts[paper]
            >= max_per_paper
        ):
            continue

        selected.append({
            "paper": paper,
            "page": chunk["page"],
            "text": chunk["text"],
            "score": float(
                scores[index]
            )
        })

        paper_counts[paper] += 1

        if len(selected) >= k:
            break

    return selected, query_vector


retrieved, query_embedding = retrieve_chunks(
    query=RETRIEVAL_QUERY,
    k=TOP_K,
    max_per_paper=MAX_CHUNKS_PER_PAPER,
    filter_reference_material=(
        FILTER_REFERENCE_MATERIAL
    )
)


if not retrieved:
    raise ValueError(
        "No chunks were retrieved. "
        "Try setting FILTER_REFERENCE_MATERIAL = False."
    )

In [16]:
# Convert the research question into an embedding

query_response = client.embeddings.create(
    model=EMBED_MODEL,
    input=QUESTION
)

query_vector = np.array(
    query_response.data[0].embedding,
    dtype=np.float32
)

query_vector /= np.linalg.norm(query_vector)

# Rank chunks by semantic similarity
scores = embedding_matrix @ query_vector
top_indices = np.argsort(scores)[::-1][:TOP_K]

retrieved = [
    {
        "paper": chunks[i]["paper"],
        "page": chunks[i]["page"],
        "text": chunks[i]["text"],
        "score": float(scores[i])
    }
    for i in top_indices
]

In [ ]:
# Review the retrieved evidence

for n, item in enumerate(retrieved, start=1):
    print(
        f"\n[S{n}] {os.path.basename(item['paper'])} "
        f"| page {item['page']} "
        f"| score {item['score']:.4f}"
    )
    print(item["text"][:900])

### Step 5. Generate a Grounded Answer

Combine the retrieved evidence and ask the model to answer using only that evidence.

In [ ]:
# Combine the retrieved passages into one context

retrieved_context = "\n\n".join(
    f"[S{n}] {os.path.basename(item['paper'])}, page {item['page']}\n"
    f"{item['text']}"
    for n, item in enumerate(retrieved, start=1)
)

instructions = """
Answer the research question using only the retrieved evidence.

- Distinguish each paper's own empirical findings from prior studies cited in the paper.
- Cite claims using the source markers [S1], [S2], and so on.
- Do not use outside knowledge.
- If the evidence is insufficient, say so.
"""

response = client.responses.create(
    model=ANSWER_MODEL,
    instructions=instructions,
    input=(
        f"Research question:\n{QUESTION}\n\n"
        f"Retrieved evidence:\n{retrieved_context}"
    )
)

print(response.output_text)